In [ ]:
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.ticker import MaxNLocator
import jax
import jax.numpy as jnp
from functools import partial
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401


mpl.rcParams.update({
    "font.family": "serif",
    "mathtext.fontset": "cm",
    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 18,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "figure.dpi": 150,
    "savefig.dpi": 350,
})


r1, r2 = 0.0, 1.0
lambda_reg = 1.0
theta_lo, theta_hi = -50.0, 50.0
grid_n = 50

alpha = 0.1
prior = jnp.array([1.0, 1.0])

cmap = plt.get_cmap("viridis")


def f_reg_tsallis(pi, a):
    return (pi**a - a*pi + a - 1) / (a * (a - 1))

def neg_entropy(pi, eps=1e-12):
    pi_c = np.clip(pi, eps, 1.0)
    return pi_c * np.log(pi_c)

theta1 = np.linspace(theta_lo, theta_hi, grid_n)
theta2 = np.linspace(theta_lo, theta_hi, grid_n)
T1, T2 = np.meshgrid(theta1, theta2)


m = np.maximum(T1, T2)
exp1 = np.exp(T1 - m)
exp2 = np.exp(T2 - m)
Z = exp1 + exp2
pi1_soft = exp1 / Z
pi2_soft = exp2 / Z


@partial(jax.jit, static_argnums=(2,))
def f_tsallis_softmax_jax(theta, prior, alpha, eps=1e-6, max_iter=200):
    j_star = jnp.argmax(theta)
    theta_max = jnp.max(theta)
    prior_star = prior[j_star]

    tau_min = theta_max - ((1 / prior_star) ** (alpha - 1) - 1) / (alpha - 1)
    tau_max = theta_max - ((1 / jnp.sum(prior)) ** (alpha - 1) - 1) / (alpha - 1)

    tau = (tau_min + tau_max) / 2
    p_tau = prior * (1 + (alpha - 1) * (theta - tau)) ** (1 / (alpha - 1))
    phi_tau = jnp.sum(p_tau) - 1

    def cond(s):
        it, _, _, phi = s
        return (jnp.abs(phi) > eps) & (it < max_iter)

    def body(s):
        it, tmin, tmax, phi = s
        mid = (tmin + tmax) / 2
        tmin2, tmax2 = jax.lax.cond(phi < 0, lambda: (tmin, mid), lambda: (mid, tmax))
        mid = (tmin2 + tmax2) / 2
        p = prior * (1 + (alpha - 1) * (theta - mid)) ** (1 / (alpha - 1))
        phi = jnp.sum(p) - 1
        return it + 1, tmin2, tmax2, phi

    _, tmin, tmax, _ = jax.lax.while_loop(cond, body, (0, tau_min, tau_max, phi_tau))
    tau = (tmin + tmax) / 2
    return prior * (1 + (alpha - 1) * (theta - tau)) ** (1 / (alpha - 1))

grid = jnp.stack([T1.ravel(), T2.ravel()], axis=1)
P = jax.vmap(lambda th: f_tsallis_softmax_jax(th, prior, alpha))(grid)
pi1_t = np.asarray(P[:, 0]).reshape(T1.shape)
pi2_t = np.asarray(P[:, 1]).reshape(T1.shape)


V_soft_ent = (
    pi1_soft * r1 + pi2_soft * r2
    - lambda_reg * (neg_entropy(pi1_soft) + neg_entropy(pi2_soft))
)

V_soft_tsreg = (
    pi1_soft * r1 + pi2_soft * r2
    - lambda_reg * (f_reg_tsallis(pi1_soft, alpha) + f_reg_tsallis(pi2_soft, alpha))
)

V_tsparam_ent = (
    pi1_t * r1 + pi2_t * r2
    - lambda_reg * (neg_entropy(pi1_t) + neg_entropy(pi2_t))
)

V_tsparam_tsreg = (
    pi1_t * r1 + pi2_t * r2
    - lambda_reg * (f_reg_tsallis(pi1_t, alpha) + f_reg_tsallis(pi2_t, alpha))
)


def style_ax(ax):
    ax.view_init(elev=15, azim=25)
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.pane.set_facecolor((1, 1, 1, 0))
        axis.pane.set_edgecolor((0, 0, 0, 0.15))
        axis.set_major_locator(MaxNLocator(nbins=4, integer=True))
    ax.grid(True, linewidth=0.4, alpha=0.25)
    ax.set_xlabel(r"$\theta_1$", labelpad=3)
    ax.set_ylabel(r"$\theta_2$", labelpad=3)

def save_single_surface(V, out_pdf, out_png=None, pad=0.03):
    vmin, vmax = float(V.min()), float(V.max())
    norm = colors.Normalize(vmin=vmin, vmax=vmax)
    zlim = (vmin - pad*(vmax - vmin),
            vmax + pad*(vmax - vmin))

    fig = plt.figure(figsize=(3.6, 3.1))
    ax = fig.add_subplot(1, 1, 1, projection="3d")
    style_ax(ax)
    ax.set_zlim(*zlim)
    ax.set_title("")  # no title

    ax.plot_surface(T1, T2, V, cmap=cmap, norm=norm,
                    rcount=140, ccount=140, alpha=0.85)
    ax.contourf(T1, T2, V, zdir="z", offset=zlim[0],
                levels=20, cmap=cmap, norm=norm, alpha=0.40)

    ax.set_zlabel(r"$\tilde V_\lambda(\theta)$", labelpad=8)

    plt.subplots_adjust(left=0.00, right=1.00, bottom=0.00, top=1.00)

    fig.savefig(out_pdf, bbox_inches="tight")
    if out_png is not None:
        fig.savefig(out_png, bbox_inches="tight")
    plt.close(fig)

os.makedirs("loss_landscape", exist_ok=True)

save_single_surface(
    V_soft_ent,
    out_pdf=f"loss_landscape/landscape_softmax_entropy.pdf",
    #out_png=f"loss_landscape/landscape_softmax_entropy.png",
)
save_single_surface(
    V_soft_tsreg,
    out_pdf=f"loss_landscape/landscape_softmax_tsallisreg_alpha{alpha}.pdf",
    #out_png=f"loss_landscape/landscape_softmax_tsallisreg_alpha{alpha}.png",
)
save_single_surface(
    V_tsparam_ent,
    out_pdf=f"loss_landscape/landscape_tsallisparam_alpha{alpha}_entropy.pdf",
    #out_png=f"loss_landscape/landscape_tsallisparam_alpha{alpha}_entropy.png",
)
save_single_surface(
    V_tsparam_tsreg,
    out_pdf=f"loss_landscape/landscape_tsallisparam_alpha{alpha}_tsallisreg_alpha{alpha}.pdf",
    #out_png=f"loss_landscape/landscape_tsallisparam_alpha{alpha}_tsallisreg_alpha{alpha}.png",
)

print("Saved 4 PDFs (and PNGs) to: loss_landscape/")


Saved 4 PDFs (and PNGs) to: loss_landscape/
